In [1]:
import pandas as pd

In [2]:
file_path = "E:/AI_ML_Learning/data/Faqs.csv"
df = pd.read_csv(file_path)

In [3]:
print(df.head())
print(df.shape)
print(df.columns.tolist())

                                    question  \
0                How do I reset my password?   
1           How can I register for a course?   
2              How do I pay my tuition fees?   
3  What should I do if I forget my username?   
4           How can I check my exam results?   

                                              answer  
0  Click Forgot Password on the login page and fo...  
1  Log in to your student portal, open Course Reg...  
2  Open the payments section of the student porta...  
3  Contact student support to recover your username.  
4  Log in to the student portal and open the Resu...  
(10, 2)
['question', 'answer']


In [4]:
print(df.isnull().sum())

question    0
answer      0
dtype: int64


In [5]:
df = df.dropna(subset=["question", "answer"])

In [6]:
df["question"] = df["question"].str.strip()
df["answer"] = df["answer"].str.strip()

In [7]:
df = df[
    (df["question"] != "") &
    (df["answer"] != "")
]

# Reset row numbering
df = df.reset_index(drop=True)

print("Cleaned dataset shape:", df.shape)
print(df.head())

Cleaned dataset shape: (10, 2)
                                    question  \
0                How do I reset my password?   
1           How can I register for a course?   
2              How do I pay my tuition fees?   
3  What should I do if I forget my username?   
4           How can I check my exam results?   

                                              answer  
0  Click Forgot Password on the login page and fo...  
1  Log in to your student portal, open Course Reg...  
2  Open the payments section of the student porta...  
3  Contact student support to recover your username.  
4  Log in to the student portal and open the Resu...  


In [8]:
def chunk_text(text, chunk_size=200, overlap=40):
    words = text.split()
    chunks = []
    start = 0

    while start < len(words):
        end = start + chunk_size
        chunk = " ".join(words[start:end])
        chunks.append(chunk)

        if end >= len(words):
            break

        start = end - overlap

    return chunks

In [9]:
faq_chunks = []

for _, row in df.iterrows():
    question = row["question"]
    answer = row["answer"]

    chunks = chunk_text(answer)

    for i, chunk in enumerate(chunks):
        faq_chunks.append({
            "id": f"faq_{_}_{i}",
            "question": question,
            "answer": answer,
            "text": f"Question: {question}\nAnswer: {chunk}"
        })

print("Total chunks:", len(faq_chunks))
print(faq_chunks[0])

Total chunks: 10
{'id': 'faq_0_0', 'question': 'How do I reset my password?', 'answer': 'Click Forgot Password on the login page and follow the instructions sent to your email.', 'text': 'Question: How do I reset my password?\nAnswer: Click Forgot Password on the login page and follow the instructions sent to your email.'}


In [10]:
import chromadb
from chromadb.utils.embedding_functions import (
    SentenceTransformerEmbeddingFunction
)

# Load the embedding model
embedding_function = SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)

# Create a persistent Chroma database
client = chromadb.PersistentClient(
    path="E:/AI_ML_Learning/chroma_db"
)

# Create or reuse a collection
collection = client.get_or_create_collection(
    name="student_faqs",
    embedding_function=embedding_function
)

print("Chroma collection ready!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Chroma collection ready!


In [11]:
collection.upsert(
    ids=[item["id"] for item in faq_chunks],
    documents=[item["text"] for item in faq_chunks],
    metadatas=[
        {
            "question": item["question"],
            "answer": item["answer"]
        }
        for item in faq_chunks
    ]
)

print("Stored chunks:", collection.count())

Stored chunks: 10


In [12]:
def search_faq(query, top_k=3):
    results = collection.query(
        query_texts=[query],
        n_results=min(top_k, collection.count())
    )

    for i, document in enumerate(results["documents"][0]):
        metadata = results["metadatas"][0][i]
        distance = results["distances"][0][i]

        print(f"\nMatch {i + 1}")
        print("Question:", metadata["question"])
        print("Answer:", metadata["answer"])
        print("Distance:", round(distance, 4))
        print("-" * 40)

In [13]:
search_faq("I forgot my login credentials")


Match 1
Question: How do I reset my password?
Answer: Click Forgot Password on the login page and follow the instructions sent to your email.
Distance: 0.374
----------------------------------------

Match 2
Question: What should I do if I forget my username?
Answer: Contact student support to recover your username.
Distance: 0.4128
----------------------------------------

Match 3
Question: How do I change my email address?
Answer: Open your account settings and update your registered email address.
Distance: 0.6835
----------------------------------------
